In [ ]:
import pandas as pd

from os import path
import re

from pymodulon.io import load_json_model
from pymodulon.plotting import *

import numpy as np
import plotly.express as px
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib_venn import venn2, venn3, venn3_circles
from upsetplot import UpSet, from_memberships, plot

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
trn = pd.read_csv('../../data/external/trn.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)

trn_supp = pd.read_csv('../../data/external/trn_full.csv', index_col=0).dropna()
trn_supp = trn_supp[["reg","gene_name", "gene_id", "effect"]]
trn_supp = trn_supp.rename({"reg":"regulator"}, axis=1)
trn_supp = trn_supp[trn_supp.gene_id.isin(ica_data.gene_table.index)]

In [ ]:
ifh1_im = set(ica_data.view_imodulon('IFH1').index)
ifh1_strong_reg = set(trn[trn.regulator == 'IFH1'].gene_id.unique())
ifh1_full_reg = set(trn_supp[trn_supp.regulator == 'IFH1'].gene_id.unique())

In [ ]:
# Create the plot
plt.figure(figsize=(8, 8))

# Draw the Venn diagram
venn = venn2([ifh1_strong_reg, ifh1_im],
             set_labels=('Strong Evidence IFH1 Regulon', 'IFH1 iModulon'),
             set_colors=('skyblue', 'salmon'), alpha=0.6)

# Add custom circles
# venn2_circles([ifh1_strong_reg, ifh1_im], linestyle='dashed', linewidth=1.0, color='gray')

# Customize fonts (optional)
for subset in venn.set_labels:
    subset.set_fontsize(15)

for label in venn.subset_labels:
    if label:  # Some labels may be None
        label.set_fontsize(12)

plt.title("IFH1 Regulatory Overlap", fontsize=20)
plt.tight_layout()
plt.savefig('images/fig2_ifh1_venn.svg', format='svg', dpi = 600)
plt.show()

In [ ]:
# Create the plot
plt.figure(figsize=(8, 8))

# Draw the Venn diagram
venn = venn3([ifh1_full_reg, ifh1_strong_reg, ifh1_im],
             set_labels=('Putative IFH1 Regulation', 'Strong Evidence IFH1 Regulon', 'IFH1 iModulon'),
             set_colors=('skyblue', 'lightgreen', 'salmon'), alpha=0.6)

# Add custom circles
venn3_circles([ifh1_full_reg, ifh1_strong_reg, ifh1_im], linestyle='dashed', linewidth=1.0, color='gray')

# Customize fonts (optional)
for subset in venn.set_labels:
    subset.set_fontsize(12)

for label in venn.subset_labels:
    if label:  # Some labels may be None
        label.set_fontsize(10)

plt.title("IFH1 Regulatory Overlap", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
set1 = ifh1_im
set2 = ifh1_strong_reg
set3 = ifh1_full_reg

set_names = ['IFH1 iModulon', 'Strong Evidence IFH1 Regulon', 'Putative IFH1 Regulation']
all_elems = set1.union(set2).union(set3)
df = pd.DataFrame([[e in set1, e in set2, e in set3] for e in all_elems], columns = set_names)
df_up = df.groupby(set_names).size()
plot(df_up, orientation='horizontal')

In [ ]:
ifg1_genes = ica_data.X.index
gene_regulation = ica_data.gene_table.loc[ifg1_genes].regulator
def reg_by_ifh1(x):
    x = str(x)
    if 'IFH1' in x:
        return 'Regulated by IFH1'
    return "No Known Regulation by IFH1"
gene_regulation = gene_regulation.apply(lambda x: reg_by_ifh1(x))
kwargs = {'groups':gene_regulation, 'colors':{'Regulated by IFH1': 'orange', "No Known Regulation by IFH1":'tab:blue'}}


import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec

# Set up a figure with two rows: one for gene weights, one for chromosomes
fig = plt.figure(figsize=(9, 6.5))
gs = GridSpec(2, 1, height_ratios=[10, 1], hspace=0.05)

# First axis: the gene weight plot
ax_main = fig.add_subplot(gs[0])

# Plot your gene weights on this axis
plot_gene_weights_chrom(ica_data, 'IFH1', ax=ax_main, **kwargs);
ax_main.set_xticks([])  # Hide x-ticks (they’ll go in the chromosome track)
ax_main.set_title("Gene Weights of the IFH1 iModulon")
ax_main.legend(
    loc='lower left',    # Bottom-left inside the plot
    bbox_to_anchor=(0, 0),  # (x=0, y=0) is bottom-left corner of the axes
    frameon=True,
    fontsize=10
)

# Second axis: chromosome track
ax_chrom = fig.add_subplot(gs[1], sharex=ax_main)
ax_chrom.set_ylim(0, 1)
ax_chrom.axis('off')  # Hide frame, ticks, etc.

# Plot chromosome rectangles here
for idx, row in ica_data.chrom.iterrows():
    chrom = idx
    start = row['location']
    width = row['length']

    rect = patches.FancyBboxPatch(
        (start, 0.3),
        width,
        0.4,
        boxstyle="round4, pad=0.01, rounding_size=0.03",
        linewidth=0.8,
        edgecolor='black',
        facecolor='lightgrey',
        fill= True
    )
    ax_chrom.add_patch(rect)

    # Add label
    ax_chrom.text(start + width / 2, 0.15, chrom, ha='center', va='top', fontsize=12)

ax_chrom.text(-1400000, .55, "Chromosomes" , ha='center', va='top', fontsize=12)


plt.xticks(rotation = 0)
plt.savefig('images/fig2_ifh1_gene_weights.svg', dpi = 600, format = 'svg', bbox_inches='tight')

plt.show()

In [ ]:
ica_data.view_imodulon("IFH1").loc['YPR132W']

In [ ]:
ifh1_regulon_genes = gene_regulation[gene_regulation == 'Regulated by IFH1'].index
ifh1_imodulons_genes = ica_data.view_imodulon("IFH1").index
ica_data.gene_table.loc[[x for x in ifh1_imodulons_genes if x not in ifh1_regulon_genes]].sort_values('gene_product')

In [ ]:
other_ifh1_regulated_genes = [x for x in ica_data.X.index if x not in ica_data.view_imodulon('IFH1').index and 'IFH1' in str(ica_data.gene_table.loc[x, 'regulator']) and
                             'RAP1' in str(ica_data.gene_table.loc[x, 'regulator'])]
ica_data.gene_table.loc[other_ifh1_regulated_genes]

In [ ]:
groups = ica_data.sample_table[((ica_data.sample_table.project == 'ebu_histone_acetyltransferase') & (ica_data.sample_table.condition.str.contains('pd')))]
groups = groups.condition.apply(lambda x: 'Post Diauxic Shift').to_dict()

groups2 = ica_data.sample_table[(ica_data.sample_table.project == 'uw_stress_response')& (ica_data.sample_table.condition.str.contains('agy4'))]
groups2 = groups2.condition.apply(lambda x: 'Salt Stress: ' + x.split('_')[-1]).to_dict()

groups = {**groups, **groups2}

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,3))
compare_activities(ica_data,'General Stress Response','IFH1', groups = groups, ax=ax)
plt.legend(loc='lower left', ncol=2, bbox_to_anchor=(1, 0))
plt.savefig('images/fig2_ifh1_gs_phase_plane.svg', format = 'svg')

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,3))
compare_activities(ica_data,'Ribosome Biogenesis Repression','IFH1', groups = groups, ax=ax)
plt.legend(loc='lower left', ncol=2, bbox_to_anchor=(1, 0))
plt.savefig('images/fig2_ifh1_rbr_phase_plane.svg', format = 'svg')

In [ ]:
plot_activities(ica_data, 'IFH1')

In [ ]:
plot_activities(ica_data, 'IFH1', projects = 'uw_stress_response')

In [ ]:
plot_activities(ica_data, 'General Stress Response')

# Fear vs Greed Examples

In [ ]:
sns.clustermap(ica_data.A.T.corr(), cmap='coolwarm', center = 0)

In [ ]:
ica_data.A.T.corr().loc['IFH1'].sort_values(ascending=False, key=abs).head(20)

In [ ]:
ica_a_corr = ica_data.A.T.corr()
fear_greed_ims = ica_a_corr.loc['IFH1'][ica_a_corr.loc['IFH1'].abs() > .7].index


fear_greed_corr = ica_data.A.T.corr().loc[fear_greed_ims, fear_greed_ims]

g = sns.clustermap(
    fear_greed_corr,
    vmin=-1, vmax=1,
    cbar_pos=(-.05, .3, 0.1, .5)
)

# Hide dendrograms but keep clustering order
g.ax_row_dendrogram.set_visible(False)

plt.savefig('images/fig2_fear_greed_clustermap.svg', format = 'svg')

In [ ]:
plot_activities(ica_data, 'Ribosome Biogenesis Repression')

In [ ]:
ica_data.sample_table[ica_data.sample_table.full_name.str.contains('h')]

In [ ]:
ica_data.M.shape

In [ ]:
import pandas as pd
import seaborn as sns

# Genes/features you're plotting
genes = [x for x in list(ifh1_im.union(ifh1_strong_reg))
         if x in ica_data.X.index]

# Correlation matrix
corr = ica_data.X.loc[genes].T.corr()

# Assign colors based on membership
def get_color(gene):
    in_im = gene in ifh1_im
    in_strong = gene in ifh1_strong_reg

    if in_im and in_strong:
        return "purple"
    elif in_im:
        return "blue"
    elif in_strong:
        return "red"
    else:
        return "gray"

colors = pd.Series(
    [get_color(g) for g in corr.index],
    index=corr.index
)

import matplotlib.patches as mpatches

g = sns.clustermap(
    corr,
    row_colors=colors,
    col_colors=colors,
    cmap="vlag"
)

handles = [
    mpatches.Patch(color="blue", label="ifh1_im"),
    mpatches.Patch(color="red", label="ifh1_strong_reg"),
    mpatches.Patch(color="purple", label="both"),
]

g.ax_heatmap.legend(
    handles=handles,
    title="Membership",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

In [ ]:
ica_data.M.loc[corr.mean().sort_values().head(9).index, 'IFH1']

In [ ]:
im_only = ifh1_im - ifh1_strong_reg
reg_only = ifh1_strong_reg - ifh1_im
overlap = ifh1_im & ifh1_strong_reg

corr = ica_data.X.T.corr()

im_only_corr = corr.loc[im_only, overlap].mean(axis=1)

In [ ]:
im_only_corr.sort_values()

In [ ]:
reg_only_corr = corr.loc[[x for x in reg_only if x in ica_data.X.index], overlap].mean(axis=1)

reg_only_corr.sort_values()

In [ ]:
all_genes = ica_data.X.index

A_ifh1 = ica_data.A.loc["IFH1"]

all_activity_corr = pd.Series({
    g: pearsonr(ica_data.X.loc[g], A_ifh1)[0]
    for g in all_genes
})

all_weights = ica_data.M["IFH1"].loc[all_genes]

all_df = pd.DataFrame({
    "weight": all_weights,
    "activity_corr": all_activity_corr
})

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))

# background (all genes)
sns.scatterplot(
    data=all_df,
    x="weight",
    y="activity_corr",
    color="lightgrey",
    alpha=0.2,
    s=10,
    linewidth=0
)

# overlay selected genes
sns.scatterplot(
    data=plot_df,
    x="weight",
    y="activity_corr",
    hue="group",
    alpha=0.9
)

plt.axvline(0, color="grey", lw=1, ls="--")
plt.axhline(0, color="grey", lw=1, ls="--")

plt.xlabel("IFH1 ICA weight (M)")
plt.ylabel("Correlation with IFH1 activity (A)")
plt.legend(title="IFH1 gene set", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

# Analysis of FHL1 Motif

In [ ]:
import xml.etree.ElementTree as ET

# 1. Parse the XML file into an ElementTree object
tree = ET.parse('../motif_search/xstreme/IFH1/meme_out/meme.xml')

# 2. Get the root element of the XML structure
root = tree.getroot()


seq_lookup = {}

for seq in root.findall(".//sequence"):
    seq_lookup[seq.attrib["id"]] = seq.attrib["name"]


# 3. Access data from the root
print(f"Root Tag: {root.tag}")                       # Output: catalog

motif2 = root.find(".//motif[@id='motif_2']")
sites = motif2.find("contributing_sites")


rows = []

for site in motif2.find("contributing_sites"):
    seq_id = site.attrib["sequence_id"]

    rows.append({
        "sequence_name": seq_lookup[seq_id],
        "sequence_id": seq_id,
        "position": int(site.attrib["position"]),
        "strand": site.attrib["strand"],
        "pvalue": float(site.attrib["pvalue"])
    })

df = pd.DataFrame(rows)
df

In [ ]:
fhl1_motif_genes = df.sequence_name.values

In [ ]:
trn[(trn.regulator.isin(['RAP1', 'IFH1', 'FHL1'])) & (trn.gene_id.isin(fhl1_motif_genes))].gene_id.unique()

In [ ]:
motif_genes = set(df.sequence_name)

ifh1_targets = set(
    trn.loc[trn.regulator == "IFH1", "gene_id"]
)

fhl1_targets = set(
    trn.loc[trn.regulator == "FHL1", "gene_id"]
)

rap1_targets = set(
    trn.loc[trn.regulator == "RAP1", "gene_id"]
)

print("Motif genes:", len(motif_genes))

print("Overlap with IFH1:",
      len(motif_genes & ifh1_targets))

print("Overlap with FHL1:",
      len(motif_genes & fhl1_targets))

print("Overlap with RAP1:",
      len(motif_genes & rap1_targets))

In [ ]:
known_genes = (
    ifh1_targets |
    fhl1_targets |
    rap1_targets
)

known_motif_genes = motif_genes & known_genes
novel_motif_genes = motif_genes - known_genes

print("Known motif genes:", len(known_motif_genes))
print("Novel motif genes:", len(novel_motif_genes))

In [ ]:
imod_genes = set(
    ica_data.view_imodulon("IFH1").index
)

motif_genes = set(df.sequence_name)

missing_motif = imod_genes - motif_genes

print("iModulon genes:", len(imod_genes))
print("Motif genes:", len(motif_genes))
print("iModulon genes without motif:", len(missing_motif))

In [ ]:
fraction = len(motif_genes & imod_genes) / len(imod_genes)

print(f"{fraction:.1%} of IFH1 iModulon genes contain motif_2")

In [ ]:
summary = pd.DataFrame({
    "category": [
        "IFH1 iModulon genes",
        "Motif genes",
        "Motif ∩ IFH1 targets",
        "Motif ∩ FHL1 targets",
        "Motif ∩ RAP1 targets",
        "Novel motif genes",
        "iModulon genes without motif"
    ],
    "count": [
        len(imod_genes),
        len(motif_genes),
        len(motif_genes & ifh1_targets),
        len(motif_genes & fhl1_targets),
        len(motif_genes & rap1_targets),
        len(novel_motif_genes),
        len(missing_motif)
    ]
})

summary

In [ ]:
motif_genes = set(df.sequence_name)

ifh1_targets = set(
    trn.loc[trn.regulator == "IFH1", "gene_id"]
)

ifh1_without_motif = ifh1_targets - motif_genes

print("IFH1 targets in TRN:", len(ifh1_targets))
print("IFH1 targets with motif:", len(ifh1_targets & motif_genes))
print("IFH1 targets without motif:", len(ifh1_without_motif))

In [ ]:
imod_genes = set(ica_data.view_imodulon("IFH1").index)

ifh1_trn_not_imod = ifh1_targets - imod_genes

imod_not_trn = imod_genes - ifh1_targets

ifh1_imod_targets = ifh1_targets & imod_genes

missed_by_motif = ifh1_imod_targets - motif_genes

print(
    f"{len(missed_by_motif)} IFH1-regulated genes are in the IFH1 iModulon "
    f"but do not contain motif_2"
)

In [ ]:
corr = ica_data.X.T.corr()

missing_motif_corr = corr.loc[list(missing_motif), list(imod_genes - missing_motif)]

In [ ]:
missing_motif_corr.median().median()

In [ ]:
corr.loc[list(imod_genes - missing_motif), list(imod_genes - missing_motif)].median().median()